# Audition model outputs — hear predicted drum dynamics

Pick a drum performance from E-GMD, run a trained model to predict every note's
**velocity** from structure/timing only, and listen to the result next to the
original human performance.

**How to use**
1. Set `MODEL`, `SPLIT`, and `SAMPLE` in the config cell below.
2. Run all cells. Audio players appear at the bottom.

**Requirements**: run this notebook with the repo `.venv` kernel; needs `fluidsynth`
(system lib) and the soundfont at `sf/big/FluidR3_GM.sf2`.

> ⚠️ **When you change `MODEL`, restart the kernel and run all.** LightGBM and
> PyTorch each load their own OpenMP runtime; mixing both in one kernel process
> can crash on macOS. Each kernel run should touch only one model.

Prereqs produced by the training scripts (Plan A / Plan B):
`data/processed/lightgbm_model.joblib`, `data/processed/transformer_best.pt`,
`data/processed/transformer_meta.json`.

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────
MODEL = "transformer"     # "transformer" (Plan B) or "lightgbm" (Plan A)
SPLIT = "test"            # "train" | "validation" | "test"
SAMPLE = None             # None -> random pick; or an int N -> the N-th file in the split
BEAT_TYPE = None          # None -> any; or "beat" / "fill"
SEED = 42

In [ ]:
# ── Setup: paths, light imports, pick a sample ────────────────────────────
import os, sys, warnings
sys.path.insert(0, "..")          # make the drumhumanizer package importable from notebooks/
warnings.simplefilter("ignore")

import numpy as np
import pandas as pd

# NOTE: only torch-free helpers are imported here. The heavy, model-specific
# imports (torch OR lightgbm) happen in the inference cell, one at a time.
from drumhumanizer.midi import load_note_array
from drumhumanizer.features import build_note_features
from drumhumanizer.metrics import mae, rmse
from drumhumanizer.playback import play_midi_file, play_midi_notes, set_soundfont

# repo-relative locations (cwd is notebooks/)
BASE = os.path.join("..", "data", "e-gmd", "e-gmd-v1.0.0")
PROC = os.path.join("..", "data", "processed")
set_soundfont(os.path.join("..", "sf", "big", "FluidR3_GM.sf2"))

csv = pd.read_csv(os.path.join(BASE, "e-gmd-v1.0.0.csv"))
pool = csv[csv.split == SPLIT]
if BEAT_TYPE:
    pool = pool[pool.beat_type == BEAT_TYPE]
pool = pool.reset_index(drop=True)
idx = np.random.RandomState(SEED).randint(len(pool)) if SAMPLE is None else (SAMPLE % len(pool))
meta = pool.iloc[idx].to_dict()
midi_path = os.path.join(BASE, meta["midi_filename"])
print(f"[{SPLIT} #{idx}] {meta['id']}")
print(f"  style={meta['style']}  bpm={meta['bpm']}  time_sig={meta['time_signature']}  beat_type={meta['beat_type']}")

In [ ]:
# ── Load the performance and build its structural features ────────────────
na = load_note_array(midi_path)
feats = build_note_features(na, meta)

# build_note_features emits rows in onset-sorted order; keep the mapping so we
# can write predictions back onto the note array (order is irrelevant to playback,
# which is time-based, but we align to true velocities for the per-track metrics).
order = np.argsort(na["onset_sec"], kind="stable")
true_vel = na["velocity"][order].astype(float)
print(f"{len(na)} notes")

In [ ]:
# ── Inference: load ONLY the selected model's stack, predict velocities ────
if MODEL == "lightgbm":
    import joblib
    saved = joblib.load(os.path.join(PROC, "lightgbm_model.joblib"))
    m, cats = saved["model"], saved["cat_categories"]
    X = feats.drop(columns=saved["drop"])
    for c in saved["cat"]:                       # align categoricals to train levels
        X[c] = X[c].astype("category").cat.set_categories(cats[c])
    pred = m.predict(X, num_iteration=saved["best_iteration"])

elif MODEL == "transformer":
    import json
    import torch
    from drumhumanizer.model import VelocityTransformer
    from drumhumanizer.seqdata import build_split_tensors, scatter_predictions
    sc = json.load(open(os.path.join(PROC, "transformer_meta.json")))
    gv = {k: int(v) for k, v in sc["genre_vocab"].items()}
    ck = torch.load(os.path.join(PROC, "transformer_best.pt"), map_location="cpu")
    model = VelocityTransformer(n_genres=len(gv) + 1)
    model.load_state_dict(ck["best_model"])
    model.eval()
    t = build_split_tensors(feats, gv, sc["bpm_mean"], sc["bpm_std"])
    with torch.no_grad():
        y = model(t["voice_idx"], t["genre_idx"], t["num_feats"], t["pad_mask"])
    pred = scatter_predictions(t["row_idx"], y, t["pad_mask"], len(feats))

else:
    raise ValueError(f"unknown MODEL {MODEL!r}")

pred = np.clip(np.rint(pred), 0, 127)            # MIDI velocities are ints in [0, 127]
print(f"{MODEL}: per-track MAE {mae(true_vel, pred):.2f}  RMSE {rmse(true_vel, pred):.2f}")
print(f"  predicted vel std {pred.std():.1f}   (true {true_vel.std():.1f})")

In [ ]:
# ── Write predicted velocities onto a copy of the note array ──────────────
na_pred = na[order].copy()
na_pred["velocity"] = pred.astype(na["velocity"].dtype)

## ▶️ Predicted dynamics (`MODEL`)

The same notes and timing as the original, but with velocities the model predicted
from structure alone.

In [ ]:
play_midi_notes(na_pred, is_drums=True)

## ▶️ Original human performance (reference)

The real recorded velocities, loaded straight from the MIDI file with `play_midi_file`.

In [ ]:
play_midi_file(midi_path, is_drums=True)

## ▶️ Flat / de-humanized reference

Every note at a constant velocity — what the input to a humanizer sounds like.
The predicted version above is the model's attempt to restore dynamics to this.

In [ ]:
na_flat = na[order].copy()
na_flat["velocity"] = np.full(len(na_flat), 80, dtype=na["velocity"].dtype)
play_midi_notes(na_flat, is_drums=True)